# CONFIG FOR PATHS 

In [32]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import Optional
import gzip
from config import PATHS

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────


# PATHS = {
#     # inputs
#     "snp_deltas":          "parquet/clinvar_new_deltas.parquet",
#     "indel_deltas":        "parquet/clinvar_indel_deltas.parquet",
#     "variant_summary":     "variant_summary.txt.gz",
#     "submission_summary":  "submission_summary.txt.gz",
#     "snp_z_thresholds":    "thresholds_snps.csv",
#     "indel_z_thresholds":  "thresholds_indels.csv",
#     # MANE and Promoter processed
#     # "gencode_gtf":         "metadata/gencode.v49.annotation.gff3.gz",  # adjust version
#     "MANE_processed":      "metadata/MANE_processed.csv",
#     "Promoter_processed":  "metadata/Promoter_processed.csv",
#     # outputs
#     "snp_strict":          "parquet/snps_strict_test.parquet",
#     "indel_strict":        "parquet/indels_strict_test.parquet",
#     "snp_annotated":       "parquet/snps_annotated_test.parquet",
#     "indel_annotated":     "parquet/indels_annotated_test.parquet",
#     "indel_annotated_signaled": "parquet/annotated_indels_signaled.parquet",
#     "snp_annotated_signaled": "parquet/annotated_snps_signaled.parquet",
#     # trackes_metadata
#     "tracks_metadata":     "metadata/functional_tracks_metadata_human.csv"
# }



# KEYS IN THE "attribute" column of GTF

# MAIN PIPELINE: from model's deltas to filtered (Rationale and review status), annotated parquet

# Extracting and aggregating MLM, BED, BW signals

In [18]:
# ──────────────────────────────────────────────────────────────────────────────
# FAST VECTORIZED SIGNAL EXTRACTION FOR LLM
# ──────────────────────────────────────────────────────────────────────────────
#
# Replaces row-wise df.apply() with vectorized NumPy operations.
# Expected speedup: 50-100x on numeric aggregation, 5-10x on string formatting.
# ──────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from typing import Optional, Dict, List, Tuple
from config import PATHS


# =============================================================================
# CORE VECTORIZED HELPERS
# =============================================================================

def _vectorized_top_k_indices(matrix: np.ndarray, k: int) -> np.ndarray:
    """
    Get top-k column indices per row by absolute value.
    Uses argpartition (O(n)) instead of full argsort (O(n log n)).
    
    Args:
        matrix: (n_rows, n_cols) array of values (already filled, no NaN)
        k: number of top signals to extract
    
    Returns:
        (n_rows, k) array of column indices, sorted by descending |value|
    """
    n_rows, n_cols = matrix.shape
    k_actual = min(k, n_cols)
    abs_matrix = np.abs(matrix)
    
    if k_actual >= n_cols:
        # Just argsort the whole thing
        top_indices = np.argsort(-abs_matrix, axis=1)[:, :k_actual]
        return top_indices
    
    # argpartition: O(n) to get top-k (unsorted)
    top_indices = np.argpartition(abs_matrix, -k_actual, axis=1)[:, -k_actual:]
    
    # Sort just the k elements per row (O(k log k) per row)
    rows = np.arange(n_rows)[:, None]
    top_vals = abs_matrix[rows, top_indices]
    sort_within = np.argsort(-top_vals, axis=1)
    top_indices_sorted = np.take_along_axis(top_indices, sort_within, axis=1)
    
    return top_indices_sorted


def _vectorized_top_k_signed(matrix: np.ndarray, k: int, direction: str) -> np.ndarray:
    """
    Get top-k column indices for gains (direction='gain') or losses (direction='loss').
    
    For gains: top-k by descending value where value > 0.01
    For losses: top-k by ascending value where value < -0.01
    
    Returns:
        (n_rows, k) array of column indices. Padded with -1 for rows with fewer signals.
    """
    n_rows, n_cols = matrix.shape
    k_actual = min(k, n_cols)
    
    if direction == 'gain':
        # Mask non-gains, sort descending
        work = matrix.copy()
        work[work <= 0.01] = -np.inf  # push non-gains to bottom
        top_indices = np.argsort(-work, axis=1)[:, :k_actual]
        # Mark invalid entries
        rows = np.arange(n_rows)[:, None]
        vals = matrix[rows, top_indices]
        top_indices[vals <= 0.01] = -1
    else:  # loss
        work = matrix.copy()
        work[work >= -0.01] = np.inf  # push non-losses to bottom
        top_indices = np.argsort(work, axis=1)[:, :k_actual]
        rows = np.arange(n_rows)[:, None]
        vals = matrix[rows, top_indices]
        top_indices[vals >= -0.01] = -1
    
    return top_indices


# =============================================================================
# STRING FORMATTING (BULK)
# =============================================================================

def _format_signal_strings(
    matrix: np.ndarray,
    ref_matrix: np.ndarray,
    top_k_indices: np.ndarray,
    feature_names: np.ndarray,
    include_ref: bool = True
) -> List[str]:
    """
    Build formatted signal strings for all rows at once.
    
    Format: "feature=delta(ref=X); feature2=delta2(ref=Y); ..."
    
    Args:
        matrix: (n_rows, n_cols) delta values
        ref_matrix: (n_rows, n_cols) reference values (may contain NaN)
        top_k_indices: (n_rows, k) indices into columns. -1 means skip.
        feature_names: (n_cols,) array of feature name strings
        include_ref: whether to include reference values
    
    Returns:
        List of formatted strings, one per row
    """
    n_rows, k = top_k_indices.shape
    results = []
    
    for i in range(n_rows):
        parts = []
        for j in range(k):
            col_idx = top_k_indices[i, j]
            if col_idx == -1:
                continue
            d = matrix[i, col_idx]
            if d == 0 and not include_ref:
                continue
            
            part = f"{feature_names[col_idx]}={d:.3f}"
            if include_ref:
                r = ref_matrix[i, col_idx]
                if not np.isnan(r):
                    part += f"(ref={r:.3f})"
            parts.append(part)
        
        results.append("; ".join(parts) if parts else "None")
    
    return results


def _format_bw_signal_strings(
    matrix: np.ndarray,
    ref_matrix: np.ndarray,
    top_k_indices: np.ndarray,
    descriptions: np.ndarray,
) -> List[str]:
    """
    Build formatted BW signal strings with metadata descriptions.
    Format: "tissue|biosample|assay=delta(ref=X); ..."
    """
    n_rows, k = top_k_indices.shape
    results = []
    
    for i in range(n_rows):
        parts = []
        for j in range(k):
            col_idx = top_k_indices[i, j]
            if col_idx == -1:
                continue
            d = matrix[i, col_idx]
            
            part = f"{descriptions[col_idx]}={d:.3f}"
            r = ref_matrix[i, col_idx]
            if not np.isnan(r):
                part += f"(ref={r:.3f})"
            parts.append(part)
        
        results.append("; ".join(parts) if parts else "None")
    
    return results


# =============================================================================
# METADATA HELPERS
# =============================================================================

def _build_metadata_lookup(metadata_df: Optional[pd.DataFrame]) -> Dict[str, Dict]:
    """Build file_id -> metadata dict."""
    if metadata_df is None:
        return {}
    
    lookup = {}
    for _, row in metadata_df.iterrows():
        file_id = row.get('file_id', '')
        lookup[file_id] = {
            'biosample_type': row.get('biosample_type', ''),
            'tissue': row.get('tissue', ''),
            'assay': row.get('assay', ''),
            'experiment_target': row.get('experiment_target', ''),
            'dataset': row.get('dataset', '')
        }
    return lookup


def _build_bw_descriptions(bw_delta_cols: List[str], metadata_lookup: Dict) -> np.ndarray:
    """Build description array for BW tracks."""
    descriptions = []
    for col in bw_delta_cols:
        track_id = col.replace('D_BW_', '')
        meta = metadata_lookup.get(track_id, {})
        if meta:
            desc = f"{meta.get('tissue', 'Unknown')}|{meta.get('biosample_type', '')}|{meta.get('assay', '')}"
            if meta.get('experiment_target'):
                desc += f"|{meta['experiment_target']}"
        else:
            desc = track_id
        descriptions.append(desc)
    return np.array(descriptions)


# =============================================================================
# MLM EXTRACTION (VECTORIZED)
# =============================================================================

def _extract_mlm_columns(df: pd.DataFrame, variant_type: str) -> Tuple[pd.DataFrame, List[str]]:
    """Extract MLM columns and return as sub-DataFrame."""
    if variant_type == 'snp':
        mlm_cols = ['REF_5mer', 'ALT_5mer', 'LLR', 'MLM_Prior', 'MLM_Delta']
    else:
        mlm_cols = [
            'LLR', 'MLM_Prior', 'MLM_Delta', 'REF_5mer', 'ALT_5mer',
            'MLM_KL_mean', 'MLM_KL_max', 'MLM_logprob_ref', 'MLM_logprob_alt',
            'MLM_logprob_delta', 'EMB_cosine_dist', 'EMB_l2_dist',
            'EMB_max_pos_dist', 'EMB_mean_pos_dist'
        ]
    
    existing_cols = [c for c in mlm_cols if c in df.columns]
    return df[existing_cols] if existing_cols else pd.DataFrame(index=df.index), existing_cols


def _format_mlm_summary_strings(df: pd.DataFrame, variant_type: str) -> List[str]:
    """Vectorized MLM summary string construction."""
    n = len(df)
    results = [""] * n
    
    # Define which columns go into the summary and their format
    core_cols = [
        ('LLR', 'LLR', '.3f'),
        ('MLM_Delta', 'MLM_Delta', '.3f'),
        ('MLM_Prior', 'MLM_Prior', '.3f'),
        ('MLM_logprob_delta', 'LogProb_Delta', '.1f'),
        ('MLM_logprob_ref', 'LogProb_Ref', '.1f'),
        
    ]
    
    indel_cols = [
        ('EMB_cosine_dist', 'EMB_cos', '.3f'),
        ('MLM_KL_max', 'KL_max', '.3f'),
        ('MLM_KL_mean', 'KL_mean', '.3f'),
    ]
    
    cols_to_format = core_cols + (indel_cols if variant_type == 'indel' else [])
    
    # Pre-extract arrays
    arrays = {}
    for col_name, _, _ in cols_to_format:
        if col_name in df.columns:
            arrays[col_name] = df[col_name].values
    
    for i in range(n):
        parts = []
        for col_name, label, fmt in cols_to_format:
            if col_name in arrays:
                val = arrays[col_name][i]
                if not pd.isna(val):
                    parts.append(f"{label}={val:{fmt}}")
        results[i] = "; ".join(parts) if parts else "None"
    
    return results


# =============================================================================
# MAIN FAST EXTRACTION
# =============================================================================

def extract_signals_comprehensive_fast(
    df: pd.DataFrame,
    delta_cols: List[str],
    metadata_df: Optional[pd.DataFrame] = None,
    k: int = 5,
    variant_type: str = 'snp'
) -> pd.DataFrame:
    """
    Vectorized replacement for row-wise extract_signals_comprehensive.
    Operates on entire DataFrame at once using NumPy.
    
    Args:
        df: Full variant DataFrame
        delta_cols: List of delta column names (D_BED_*, D_BW_*)
        metadata_df: Track metadata DataFrame
        k: Number of top signals per category
        variant_type: 'snp' or 'indel'
    
    Returns:
        DataFrame with signal columns to join back
    """
    n_rows = len(df)
    result_df = pd.DataFrame(index=df.index)
    
    # -----------------------------------------------------------------
    # Separate column types
    # -----------------------------------------------------------------
    bed_delta_cols = [c for c in delta_cols if c.startswith('D_BED_')]
    bw_delta_cols = [c for c in delta_cols if c.startswith('D_BW_')]
    
    # -----------------------------------------------------------------
    # BED SIGNALS
    # -----------------------------------------------------------------
    if bed_delta_cols:
        bed_feature_names = np.array([c.replace('D_BED_', '') for c in bed_delta_cols])
        bed_ref_cols = [c.replace('D_BED_', 'REF_BED_') for c in bed_delta_cols]
        # Ensure ref cols exist
        bed_ref_cols_exist = [c for c in bed_ref_cols if c in df.columns]
        
        # Extract matrices
        bed_matrix = df[bed_delta_cols].fillna(0).values.astype(np.float64)
        
        # Build ref matrix (align columns)
        if bed_ref_cols_exist:
            bed_ref_matrix = np.full_like(bed_matrix, np.nan)
            for i, delta_col in enumerate(bed_delta_cols):
                ref_col = delta_col.replace('D_BED_', 'REF_BED_')
                if ref_col in df.columns:
                    bed_ref_matrix[:, i] = pd.to_numeric(df[ref_col], errors='coerce').values
        else:
            bed_ref_matrix = np.full_like(bed_matrix, np.nan)
        
        # Aggregate metrics (fully vectorized — no Python loops)
        abs_bed = np.abs(bed_matrix)
        result_df['BED_Max_Abs_Delta'] = abs_bed.max(axis=1)
        result_df['BED_N_Strong_Signals'] = (abs_bed > 0.3).sum(axis=1)
        
        # Top-K indices
        bed_top_abs_idx = _vectorized_top_k_indices(bed_matrix, k)
        bed_top_gain_idx = _vectorized_top_k_signed(bed_matrix, k, 'gain')
        bed_top_loss_idx = _vectorized_top_k_signed(bed_matrix, k, 'loss')
        
        # Format strings
        result_df['BED_Top_Abs'] = _format_signal_strings(
            bed_matrix, bed_ref_matrix, bed_top_abs_idx, bed_feature_names
        )
        result_df['BED_Top_Gains'] = _format_signal_strings(
            bed_matrix, bed_ref_matrix, bed_top_gain_idx, bed_feature_names
        )
        result_df['BED_Top_Losses'] = _format_signal_strings(
            bed_matrix, bed_ref_matrix, bed_top_loss_idx, bed_feature_names
        )
    else:
        result_df['BED_Max_Abs_Delta'] = 0
        result_df['BED_N_Strong_Signals'] = 0
        result_df['BED_Top_Abs'] = "None"
        result_df['BED_Top_Gains'] = "None"
        result_df['BED_Top_Losses'] = "None"
    
    # -----------------------------------------------------------------
    # BW SIGNALS (with metadata)
    # -----------------------------------------------------------------
    if bw_delta_cols:
        metadata_lookup = _build_metadata_lookup(metadata_df)
        bw_descriptions = _build_bw_descriptions(bw_delta_cols, metadata_lookup)
        bw_ref_cols = [c.replace('D_BW_', 'REF_BW_') for c in bw_delta_cols]
        
        bw_matrix = df[bw_delta_cols].fillna(0).values.astype(np.float64)
        
        bw_ref_matrix = np.full_like(bw_matrix, np.nan)
        for i, delta_col in enumerate(bw_delta_cols):
            ref_col = delta_col.replace('D_BW_', 'REF_BW_')
            if ref_col in df.columns:
                bw_ref_matrix[:, i] = pd.to_numeric(df[ref_col], errors='coerce').values
        
        # Aggregate metrics
        abs_bw = np.abs(bw_matrix)
        result_df['BW_Max_Abs_Delta'] = abs_bw.max(axis=1)
        result_df['BW_N_Strong_Signals'] = (abs_bw > 0.3).sum(axis=1)
        
        # Top-K indices
        bw_top_abs_idx = _vectorized_top_k_indices(bw_matrix, k)
        bw_top_gain_idx = _vectorized_top_k_signed(bw_matrix, k, 'gain')
        bw_top_loss_idx = _vectorized_top_k_signed(bw_matrix, k, 'loss')
        
        # Format strings
        result_df['BW_Top_Abs'] = _format_bw_signal_strings(
            bw_matrix, bw_ref_matrix, bw_top_abs_idx, bw_descriptions
        )
        result_df['BW_Top_Gains'] = _format_bw_signal_strings(
            bw_matrix, bw_ref_matrix, bw_top_gain_idx, bw_descriptions
        )
        result_df['BW_Top_Losses'] = _format_bw_signal_strings(
            bw_matrix, bw_ref_matrix, bw_top_loss_idx, bw_descriptions
        )
    else:
        result_df['BW_Max_Abs_Delta'] = 0
        result_df['BW_N_Strong_Signals'] = 0
        result_df['BW_Top_Abs'] = "None"
        result_df['BW_Top_Gains'] = "None"
        result_df['BW_Top_Losses'] = "None"
    
    # -----------------------------------------------------------------
    # MLM SIGNALS
    # -----------------------------------------------------------------
    mlm_sub, mlm_cols_used = _extract_mlm_columns(df, variant_type)
    
    # Individual MLM columns
    for col in mlm_cols_used:
        result_df[f'MLM_{col}'] = mlm_sub[col].values if col in mlm_sub.columns else np.nan
    
    # Summary string
    result_df['MLM_Summary'] = _format_mlm_summary_strings(df, variant_type)
    
    return result_df


# =============================================================================
# LLM PROMPT FORMATTER (SINGLE-ROW, ON-DEMAND)
# =============================================================================

def extract_signals_for_llm(
    row,
    delta_cols: List[str],
    metadata_df: Optional[pd.DataFrame] = None,
    k: int = 5,
    variant_type: str = 'snp'
) -> Dict:
    """
    Format signals for a SINGLE variant's LLM prompt.
    Call this at inference time, NOT during batch processing.
    
    Can work from either:
    - Pre-computed signal columns (if extract_signals_comprehensive_fast was run)
    - Raw delta columns (slower, self-contained fallback)
    """
    
    # --- Check if pre-computed columns exist ---
    if 'BED_Top_Abs' in row.index and 'BW_Top_Abs' in row.index:
        # Fast path: reformat pre-computed strings into LLM-friendly text
        bed_text = _reformat_compact_to_llm(row.get('BED_Top_Abs', 'None'), 'BED')
        bw_text = _reformat_compact_to_llm(row.get('BW_Top_Abs', 'None'), 'BW')
        mlm_text = _reformat_mlm_to_llm(row, variant_type)
        
        return {
            'BED_Signals_Text': bed_text,
            'BW_Signals_Text': bw_text,
            'MLM_Signals_Text': mlm_text,
            'Full_Signal_Block': f"""**BED Feature Changes (Top {k}):**
{bed_text}

**Epigenomic Track Changes (Top {k}):**
{bw_text}

**MLM Sequence Context:**
{mlm_text}"""
        }
    
    # --- Slow fallback: compute from raw columns ---
    # (Same logic as original, for standalone use)
    return _extract_signals_for_llm_from_raw(row, delta_cols, metadata_df, k, variant_type)


def _reformat_compact_to_llm(compact_str: str, signal_type: str) -> str:
    """Convert 'feature=delta(ref=X); ...' to verbose LLM format."""
    if compact_str == "None" or not compact_str:
        return "  None"
    
    lines = []
    for entry in compact_str.split("; "):
        # Parse: "feature=delta(ref=X)" or "feature=delta"
        if "=" not in entry:
            continue
        
        name_part, rest = entry.split("=", 1)
        
        # Check for ref value
        if "(ref=" in rest:
            delta_str, ref_part = rest.split("(ref=", 1)
            ref_str = ref_part.rstrip(")")
            lines.append(f"  - {name_part}: Δ={float(delta_str):+.3f} (REF={float(ref_str):.3f})")
        else:
            lines.append(f"  - {name_part}: Δ={float(rest):+.3f}")
    
    return "\n".join(lines) if lines else "  None"


def _reformat_mlm_to_llm(row, variant_type: str) -> str:
    """Reformat MLM columns into LLM-friendly text."""
    if variant_type == 'snp':
        cols = ['REF_5mer', 'ALT_5mer', 'LLR', 'MLM_Prior', 'MLM_Delta']
    else:
        cols = [
            'LLR', 'MLM_Prior', 'MLM_Delta', 'REF_5mer', 'ALT_5mer',
            'MLM_KL_mean', 'MLM_KL_max', 'MLM_logprob_ref', 'MLM_logprob_alt',
            'MLM_logprob_delta', 'EMB_cosine_dist', 'EMB_l2_dist',
            'EMB_max_pos_dist', 'EMB_mean_pos_dist'
        ]
    
    lines = []
    for col in cols:
        # Check both raw and prefixed versions
        val = row.get(f'MLM_{col}', row.get(col, np.nan))
        if not pd.isna(val):
            if isinstance(val, str):
                lines.append(f"  - {col}: {val}")
            else:
                lines.append(f"  - {col}: {val:.4f}")
    
    return "\n".join(lines) if lines else "  None"


def _extract_signals_for_llm_from_raw(row, delta_cols, metadata_df, k, variant_type):
    """Fallback: original row-wise extraction for single-variant use."""
    bed_delta_cols = [c for c in delta_cols if c.startswith('D_BED_')]
    bw_delta_cols = [c for c in delta_cols if c.startswith('D_BW_')]
    
    metadata_lookup = _build_metadata_lookup(metadata_df)
    
    # BED
    bed_entries = []
    for col in bed_delta_cols:
        delta = row.get(col, 0)
        if pd.isna(delta): delta = 0
        ref = row.get(col.replace('D_BED_', 'REF_BED_'), np.nan)
        feature = col.replace('D_BED_', '')
        bed_entries.append({'feature': feature, 'delta': delta,
                           'ref': ref if not pd.isna(ref) else None})
    
    bed_sorted = sorted(bed_entries, key=lambda x: abs(x['delta']), reverse=True)[:k]
    bed_lines = []
    for e in bed_sorted:
        line = f"  - {e['feature']}: Δ={e['delta']:+.3f}"
        if e['ref'] is not None: line += f" (REF={e['ref']:.3f})"
        bed_lines.append(line)
    bed_text = "\n".join(bed_lines) if bed_lines else "  None"
    
    # BW
    bw_entries = []
    for col in bw_delta_cols:
        delta = row.get(col, 0)
        if pd.isna(delta): delta = 0
        ref = row.get(col.replace('D_BW_', 'REF_BW_'), np.nan)
        track_id = col.replace('D_BW_', '')
        meta = metadata_lookup.get(track_id, {})
        if meta:
            desc = f"{meta.get('tissue', '?')} / {meta.get('biosample_type', '?')} / {meta.get('assay', '?')}"
            if meta.get('experiment_target'): desc += f" / {meta['experiment_target']}"
        else:
            desc = track_id
        bw_entries.append({'description': desc, 'delta': delta,
                          'ref': ref if not pd.isna(ref) else None})
    
    bw_sorted = sorted(bw_entries, key=lambda x: abs(x['delta']), reverse=True)[:k]
    bw_lines = []
    for e in bw_sorted:
        line = f"  - {e['description']}: Δ={e['delta']:+.3f}"
        if e['ref'] is not None: line += f" (REF={e['ref']:.3f})"
        bw_lines.append(line)
    bw_text = "\n".join(bw_lines) if bw_lines else "  None"
    
    # MLM
    mlm_text = _reformat_mlm_to_llm(row, variant_type)
    
    return {
        'BED_Signals_Text': bed_text,
        'BW_Signals_Text': bw_text,
        'MLM_Signals_Text': mlm_text,
        'Full_Signal_Block': f"""**BED Feature Changes (Top {k}):**
{bed_text}

**Epigenomic Track Changes (Top {k}):**
{bw_text}

**MLM Sequence Context:**
{mlm_text}"""
    }


# =============================================================================
# UPDATED MAIN EXECUTION
# =============================================================================

def main(variant_type_arg, PATHS):
    """
    Drop-in replacement for original main().
    
    Usage:
        from extract_signals_fast import main
        df = main("snps", PATHS)
    """
    import pyarrow.parquet as pq
    
    # --- CONFIGURATION ---
    if variant_type_arg == "indels":
        INPUT_FILE = PATHS["animals_indel_deltas"]
        METADATA_FILE = PATHS["tracks_metadata"]
        OUTPUT_FILE = PATHS["animals_indel_signaled"]
    elif variant_type_arg == "snps":
        INPUT_FILE = PATHS["animals_snp_deltas"]
        METADATA_FILE = PATHS["tracks_metadata"]
        OUTPUT_FILE = PATHS["animals_snp_signaled"]
    else:
        raise ValueError(f"Unknown variant type: {variant_type_arg}")
    
    K_SIGNALS = 10
    
    # --- LOAD DATA ---
    print("Loading data...")
    variant_df = pq.read_table(INPUT_FILE).to_pandas()
    print(f"Loaded {len(variant_df)} variants")
    
    # Load metadata
    print("Loading track metadata...")
    try:
        metadata_df = pd.read_csv(METADATA_FILE)
        print(f"Loaded metadata for {len(metadata_df)} tracks")
    except Exception as e:
        print(f"Warning: Could not load metadata ({e}). BW signals will use track IDs.")
        metadata_df = None
    
    # Identify column types
    delta_cols = [c for c in variant_df.columns if c.startswith('D_BED_') or c.startswith('D_BW_')]
    bed_delta_cols = [c for c in delta_cols if c.startswith('D_BED_')]
    bw_delta_cols = [c for c in delta_cols if c.startswith('D_BW_')]
    
    print(f"Found {len(bed_delta_cols)} BED delta columns")
    print(f"Found {len(bw_delta_cols)} BW delta columns")
    
    # Detect variant type
    variant_type = variant_type_arg


    # Ensure numeric
    print("Coercing delta columns to numeric...")
    for col in delta_cols:
        variant_df[col] = pd.to_numeric(variant_df[col], errors='coerce')
    
    # --- z-score the logprob_delta
    variant_df['MLM_logprob_delta'] = variant_df['MLM_logprob_delta'].apply(lambda x: (x - variant_df['MLM_logprob_delta'].mean()) / variant_df['MLM_logprob_delta'].std())
    variant_df['MLM_logprob_ref'] = variant_df['MLM_logprob_ref'].apply(lambda x: (x - variant_df['MLM_logprob_ref'].mean()) / variant_df['MLM_logprob_ref'].std())

    # --- FAST VECTORIZED EXTRACTION ---
    print("Extracting signals (vectorized)...")
    signal_df = extract_signals_comprehensive_fast(
        variant_df, delta_cols, metadata_df, k=K_SIGNALS, variant_type=variant_type
    )
    
    # Join results
    for col in signal_df.columns:
        variant_df[col] = signal_df[col].values
    
    variant_df.reset_index(inplace=True)
    variant_df.rename(columns={"index": "#VariationID", "rationale": "FullRationale"}, inplace=True)
    
    # --- SAVE ---
    variant_df.to_parquet(OUTPUT_FILE)
    print(f"\nResults saved to: {OUTPUT_FILE}")
    
    return variant_df


if __name__ == "__main__":
    # Example usage — define your PATHS dict before calling
    # PATHS = { "snp_annotated": "...", "tracks_metadata": "...", "snp_annotated_signaled": "..." }
    df = main("snps", PATHS)


Loading data...
Loaded 372 variants
Loading track metadata...
Loaded metadata for 7362 tracks
Found 21 BED delta columns
Found 0 BW delta columns
Coercing delta columns to numeric...
Extracting signals (vectorized)...

Results saved to: parquet/animals_snp_signaled.parquet


# DEFINING varaint and system prompts and setting up functions

In [1]:
import pandas as pd
import numpy as np

########################################################################
# ──────────────────────────────────────────────────────────────────────────────
# SYSTEM PROMPTS + VARIANT PROMPTS + FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
########################################################################


# =============================================================================
# SYSTEM PROMPTS
# =============================================================================

SYSTEM_PROMPT_SNP = """You are an expert molecular geneticist specializing in genetic variants. Your task is to evaluate whether computational signals from a genomic language model (Nucleotide Transformer) output for a SNP variant aligns or supports the human-curated ClinVar rationales (these include  rationales , consequences or annotations). Emphasis on the functional interpretation of predictions or consequences.

## INPUT DATA PER VARIANT

1. **Variant ID**: ClinVar variation identifier
2. **Region (Coarse Class)**: High-level genomic context derived from overlap annotations.
   - One of: CODING, SPLICE, UTR_5, UTR_3, PROMOTER, INTRONIC, GENIC_OTHER
   - This is a *contextual hint only*, not definitive evidence of mechanism.
   - BED and MLM signals take precedence over region when inferring mechanism.
3. **ClinVar Rationale** (FullRationale): Human-curated explanation of variant consequence
4. **BED Feature Signals**: Delta predictions (alt - ref) for genomic annotations
   - Features include: splice_donor, splice_acceptor, exon, intron, ORF, start_codon, stop_codon, 5UTR, 3UTR, promoter, enhancer, CTCF, polyA_signal, etc.
   - Format: `feature=delta(ref=X)` where delta is the change and ref is the reference prediction
5. **MLM Sequence Context**: Masked language model scores capturing sequence plausibility
   - LLR: Log-likelihood ratio log(P(ALT)/P(REF))
   - MLM_Delta: Change in model's sequence prediction confidence
   - MLM_Prior: Background sequence context score
   - LogProb_Delta: Change in sequence log probability **(Z-SCORED)**
   - LogProb_Ref: Log probability of reference sequence **(Z-SCORED)**

## CRITICAL: SIGNAL SCALES AND UNITS

The signal types operate on DIFFERENT scales. You must account for this when interpreting magnitudes:

### BED Features — Probability Scale [0, 1]
BED deltas represent changes in predicted **probabilities** of genomic annotations.
- Values are bounded: deltas range from -1.0 to +1.0
- A delta of -0.8 on a feature with ref=0.9 means the probability dropped from 90% to 10%
- **Negative delta (Loss)**: Variant disrupts the model's recognition of that genomic element
  - Example: `splice_donor=-0.8(ref=0.9)` → splice donor probability dropped from 90% to 10%
- **Positive delta (Gain)**: Variant creates or strengthens association with element
  - Example: `splice_donor=+0.6(ref=0.1)` → cryptic splice site gained (10% → 70%)


### MLM Scores — Log-Likelihood Scale (unbounded) + Z-Scored Metrics
- **LLR**: Log-likelihood ratio (alt vs ref sequence plausibility)
- **MLM_Delta**: Large magnitude indicates significant sequence perturbation
- **LogProb_Delta (Z-SCORED)**: Z-score of log probability change.
- **LogProb_Ref (Z-SCORED)**: Z-score of reference sequence log probability. Indicates how typical the reference context is

**Important**: While NT doesn't explicitly model protein structure, the MLM component could capture evolutionary sequence constraints that INCLUDE codon usage, amino acid conservation patterns, and protein-coding sequence signatures. A missense variant at a highly conserved residue will often show:
- Strong negative LLR (the alt codon/sequence is evolutionarily disfavored)
- Disruption in ORF or exon signals (even if subtle)

Therefore, MLM signals CAN provide indirect evidence for protein-level consequence through sequence conservation patterns. Do not dismiss missense variants as "NOT_APPLICABLE" if MLM signals are strong.

## YOUR OUTPUT

Return a single JSON object with these fields:

```json
{
  "explanation": "Brief reasoning (max 30 words) connecting signals to rationale",
  "concordance": "CONCORDANT|DISCORDANT",
  "signal_category": "STRONG|MODERATE|WEAK|ABSENT",
  "MLM_category": "STRONG|MODERATE|WEAK|ABSENT",
  "primary_signal_mechanism": "Inferred mechanism from signals",
  "key_signals": "Top 2-3 from each BED/MLM features driving the interpretation",
  "rationale_mechanism": "What mechanism does ClinVar describe?",
  "nt_missed": true|false,
  "notes": "Optional additional context"
}
```

## OUTPUT RULES
- DO NOT mix primary_signal_mechanism which is from input signals and rationale_mechanism derived from Clinvar rationale.
- DO NOT mix signal_category and MLM_category. signal_category is based on BED signals, and MLM on the MLM signals.


## CONCORDANCE DEFINITIONS

- **CONCORDANT**: Signals support the mechanism described in the rationale, for example:
  - Rationale says "disrupts splicing" AND BED shows splice_donor/acceptor loss (probability drop)
  - Rationale says "regulatory variant" shows tissue-relevant chromatin changes (probability drops or gains)
  - Rationale describes conserved residue AND MLM shows negative LLR
  - Rationale is missing/minimal AND signals are absent (both uninformative = concordant by default)


- **DISCORDANT**: Signals contradict or fail to support the mechanism when they SHOULD, for example:
  - Rationale explicitly describes splice disruption but NO splice signals detected (BED probability unchanged)
  - Rationale describes highly conserved position but MLM shows no sequence constraint (LLR ~ 0) or positive LLR
  - Strong signals present but don't match the described mechanism at all


## RULES
- Use expert biological reasoning
- Account for the different scales: BED are  probability [0,1], MLM is log-likelihood (unbounded), and some metrics are Z-scored
- Do NOT infer a mechanism solely from the Region field.
  -- Region provides contextual plausibility; BED/MLM signals point to mechanism.
- MLM signals provide indirect protein-level evidence through sequence conservation — do not ignore them for missense variants
- A variant can have BOTH transcript and protein effects — capture what NT detects
- Output only the JSON object, no additional text


"""


SYSTEM_PROMPT_INDEL = """You are an expert molecular geneticist specializing in genetic variants. Your task is to evaluate whether computational signals from a genomic language model (Nucleotide Transformer) output for a INDEL/DEL/INS variant aligns or supports the human-curated ClinVar rationales (these include  rationales , consequences or annotations). Emphasis on the functional interpretation of predictions or consequences.

## INPUT DATA PER VARIANT

1. **Variant ID**: ClinVar variation identifier
2. **Region (Coarse Class)**: High-level genomic context derived from overlap annotations.
   - One of: CODING, SPLICE, UTR_5, UTR_3, PROMOTER, INTRONIC, GENIC_OTHER
   - This is a *contextual hint only*, not definitive evidence of mechanism.
   - BED, BW and MLM signals take precedence over region when inferring mechanism.
3. **ClinVar Rationale** (FullRationale): Human-curated explanation of variant consequence
4. **BED Feature Signals**: Delta predictions (alt - ref) for genomic annotations
   - Features: splice_donor, splice_acceptor, exon, intron, ORF, start_codon, stop_codon, UTRs, regulatory elements
   - Format: `feature=delta(ref=X)`
5. **MLM Sequence Context** (expanded for indels):
   - LLR: Log-likelihood ratio (alt vs ref sequence plausibility)
   - MLM_Delta: Sequence prediction confidence change
   - LogProb_Delta: Change in sequence log probability **(Z-SCORED)**
   - LogProb_Ref: Log probability of reference sequence **(Z-SCORED)**
   - MLM_KL_mean: Mean KL divergence of token distributions (average local sequence disruption)
   - MLM_KL_max: Max KL divergence of token distributions (peak local sequence disruption)

## CRITICAL: SIGNAL SCALES AND UNITS

The signal types operate on DIFFERENT scales. You must account for this when interpreting magnitudes:

### BED Features — Probability Scale [0, 1]
BED deltas represent changes in predicted **probabilities** of genomic annotations.
- Values are bounded: deltas range from -1.0 to +1.0
- A delta of -0.8 on a feature with ref=0.9 means the probability dropped from 90% to 10%



### MLM — Various Scales (some Z-scored)
- **LLR**: Log-likelihood scale.
- **MLM_KL_max**: Unbounded (NOT z-scored).
- **MLM_KL_mean**: Unbounded (NOT z-scored). Average disruption across positions
- **LogProb_Delta (Z-SCORED)**: Z-score of log probability change.
- **LogProb_Ref (Z-SCORED)**: Z-score of reference sequence log probability


## INDEL-SPECIFIC INTERPRETATION

### Frameshift Detection
- Indels in coding regions often cause frameshifts leading to premature stop codons or nonsense-mediated decay
- Look for: ORF loss (BED probability drop), exon disruption, downstream stop codon effects
- BED signals may show: `ORF=-0.8, exon=-0.5` indicating loss of coding identity

### Splice Site Indels
- Look for splice_donor/acceptor losses (probability drops)

### MLM KL Divergence
- **MLM_KL_mean**: Average disruption; useful for assessing overall sequence perturbation
- Indicates the indel creates sequence context that violates learned patterns

### Protein-Level Effects via MLM
For in-frame indels affecting protein function, the MLM signals CAN capture disruption:
- negative LogProb_Delta (large negative z-score) indicates the alt sequence is evolutionarily disfavored
- These serve as proxies for protein constraint even without explicit protein modeling

## YOUR OUTPUT

Return a single JSON object:

```json
{
  "explanation": "Brief reasoning (max 30 words)",
  "concordance": "CONCORDANT|DISCORDANT",
  "signal_category": "STRONG|MODERATE|WEAK|ABSENT",
  "MLM_category": "STRONG|MODERATE|WEAK|ABSENT",
  "primary_signal_mechanism": "Inferred mechanism from NT signals (e.g., 'frameshift_ORF_loss', 'splice_disruption', 'sequence_constraint_violation')",
  "key_signals": "Top 2-3 signals driving interpretation",
  "rationale_mechanism": "What mechanism does ClinVar describe?",
  "nt_missed": true|false,
  "notes": "Optional context"
}
```

## OUTPUT RULES
- DO NOT mix primary_signal_mechanism which is from input signals and rationale_mechanism derived from Clinvar rationale.
- DO NOT mix signal_category and MLM_category. signal_category is based on BED signals, and MLM on the MLM signals.



## CONCORDANCE DEFINITIONS

- **CONCORDANT**: Signals support the mechanism, for example:
  - Frameshift rationale + strong ORF/exon probability loss
  - Splice site indel + splice signal probability loss
  - In-frame deletion at conserved region + strong negative LogProb_Delta z-score
  - Rationale missing AND signals absent (both uninformative = concordant by default)



- **DISCORDANT**: Signals contradict rationale
  - Rationale describes frameshift/splice disruption but BED probabilities
  - Strong signals present but mechanism doesn't match at all
  - Rationale describes severe effect but MLM signals are low and BED probabilities unchanged



## RULES
- Account for the different scales: BED  are  probability [0,1], MLM have their own scales
- MLM signals provide indirect protein-level evidence — high values suggest functional impact
- Consider BOTH BED (structural annotation probabilities) AND MLM (sequence context) signals
- Output only the JSON object
"""



# =============================================================================
# VARIANT PROMPT BUILDERS
# =============================================================================

def make_variant_prompt_snp(row: pd.Series) -> str:
    """
    Build a per-variant prompt for SNP concordance judgment.
    """
    def safe_get(col, default="(not available)"):
        v = row.get(col, default)
        if pd.isna(v) or v == "":
            return default
        return str(v)
    
    # Core identifiers
    variant_id = row.name if row.name else safe_get('#VariationID', 'Unknown')
    region = safe_get('region', 'Unknown')
    region_class = safe_get('region_class', 'Unknown')
    
    # ClinVar rationale
    rationale = safe_get('FullRationale')
    if rationale == "(not available)" or rationale.strip() == "":
        rationale = "(No detailed rationale provided)"
    elif len(rationale) > 1500:
        rationale = rationale[:1500] + "... [truncated]"
    
    # BED signals
    bed_abs = safe_get('BED_Top_Abs', 'None')
    bed_gains = safe_get('BED_Top_Gains', 'None')
    bed_losses = safe_get('BED_Top_Losses', 'None')
    
    # BW signals
    bw_abs = safe_get('BW_Top_Abs', 'None')
    bw_gains = safe_get('BW_Top_Gains', 'None')
    bw_losses = safe_get('BW_Top_Losses', 'None')
    
    # MLM signals
    mlm_summary = safe_get('MLM_Summary', 'None')
    
    prompt = f"""## Variant: {variant_id}

### ClinVar Rationale
{rationale}

### BED Feature Signals (Genomic Annotations)
**Top by magnitude:** {bed_abs}
**Top gains (positive delta):** {bed_gains}
**Top losses (negative delta):** {bed_losses}


### MLM Sequence Context
{mlm_summary}

---
Evaluate concordance between the NT signals and the ClinVar rationale. Return JSON only."""

    return prompt


def make_variant_prompt_indel(row: pd.Series) -> str:
    """
    Build a per-variant prompt for INDEL concordance judgment.
    Includes expanded MLM metrics relevant to indels.
    """
    def safe_get(col, default="(not available)"):
        v = row.get(col, default)
        if pd.isna(v) or v == "":
            return default
        if isinstance(v, float):
            return f"{v:.4f}"
        return str(v)
    
    # Core identifiers
    variant_id = row.name if row.name else safe_get('#VariationID', 'Unknown')
    region = safe_get('region', 'Unknown')
    indel_size = row.get('indel_size', 'Unknown')
    variant_type = row.get('variant_type', 'Unknown')
    
    # ClinVar rationale
    rationale = safe_get('FullRationale')
    if rationale == "(not available)" or rationale.strip() == "":
        rationale = "(No detailed rationale provided)"
    elif len(rationale) > 1500:
        rationale = rationale[:1500] + "... [truncated]"
    
    # BED signals
    bed_abs = safe_get('BED_Top_Abs', 'None')
    bed_gains = safe_get('BED_Top_Gains', 'None')
    bed_losses = safe_get('BED_Top_Losses', 'None')
    
    # BW signals
    bw_abs = safe_get('BW_Top_Abs', 'None')
    bw_gains = safe_get('BW_Top_Gains', 'None')
    bw_losses = safe_get('BW_Top_Losses', 'None')
    
    # MLM summary
    mlm_summary = safe_get('MLM_Summary', 'None')
    
    # Expanded MLM metrics for indels
    mlm_logprob_delta = safe_get('MLM_logprob_delta', 'N/A')
    emb_cosine = safe_get('EMB_cosine_dist', 'N/A')
    emb_l2 = safe_get('EMB_l2_dist', 'N/A')
    kl_max = safe_get('MLM_KL_max', 'N/A')
    kl_mean = safe_get('MLM_KL_mean', 'N/A')
    
    prompt = f"""## Variant: {variant_id}
**Variant Type:** {variant_type}
**Variant Size:** {indel_size}

### ClinVar Rationale
{rationale}

### BED Feature Signals (Genomic Annotations)
**Top by magnitude:** {bed_abs}
**Top gains (positive delta):** {bed_gains}
**Top losses (negative delta):** {bed_losses}


### MLM Sequence Context
**Summary:** {mlm_summary}

**Detailed Indel Metrics:**
- Log-probability delta: {mlm_logprob_delta}
- KL divergence (max): {kl_max}
- KL divergence (mean): {kl_mean}

---
Evaluate concordance between the NT signals and the ClinVar rationale. Return JSON only."""

    return prompt


def make_variant_prompt(row: pd.Series, variant_type: str = 'snp') -> str:
    """
    Dispatch to appropriate prompt builder based on variant type.
    """
    if variant_type == 'indel':
        return make_variant_prompt_indel(row)
    else:
        return make_variant_prompt_snp(row)


# =============================================================================
# PROMPT TABLE BUILDER
# =============================================================================

def build_prompts_table(df: pd.DataFrame, variant_type: str = 'snp') -> pd.DataFrame:
    """
    Add 'llm_prompt' column to dataframe.
    
    Args:
        df: DataFrame with signal columns
        variant_type: 'snp' or 'indel'
    
    Returns:
        DataFrame with added 'llm_prompt' column
    """
    out = df.copy()
    out['llm_prompt'] = out.apply(
        lambda row: make_variant_prompt(row, variant_type), 
        axis=1
    )
    return out


def get_system_prompt(variant_type: str = 'snp') -> str:
    """
    Return appropriate system prompt for variant type.
    """
    if variant_type == 'indel':
        return SYSTEM_PROMPT_INDEL
    else:
        return SYSTEM_PROMPT_SNP



# =============================================================================
# EXAMPLE USAGE
# =============================================================================

# if __name__ == "__main__":
    # Example: Load and process SNPs    
    
    # Load SNPs
    # print("Loading SNP data...")
    # snp_df = pq.read_table(PATHS["snp_annotated_signaled"]).to_pandas()
    # print(f"Loaded {len(snp_df)} SNPs")
    
    # # Build prompts
    # snp_with_prompts = build_prompts_table(snp_df, variant_type='snp')


    # # Load indels
    # print("Loading INDEL data...")
    # indel_df = pq.read_table(PATHS["indel_annotated_signaled"]).to_pandas()
    # print(f"Loaded {len(indel_df)} INDELs")

    # # Build prompts
    # indel_with_prompts = build_prompts_table(indel_df, variant_type='indel')


# A one by one API access for LLM evaluation

In [6]:
import pandas as pd
import numpy as np
import json
import time
import re
from typing import Optional, Dict, Any, List
from google import genai
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from google.api_core import exceptions

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Or load from environment
MODEL_NAME = "gemini-1.5-flash"  # Or "gemini-1.5-pro" for better quality

# =============================================================================
# API CALL WITH RETRY
# =============================================================================

_thread_local = threading.local()

def get_thread_client(api_key: str):
    # One client per thread
    if not hasattr(_thread_local, "client"):
        from google import genai
        _thread_local.client = genai.Client(api_key=api_key)
    return _thread_local.client

def call_llm_api_robust_threaded(
    user_content: str,
    system_content: str,
    model_name: str,
    api_key: str,
    max_retries: int = 10,
):

    client = get_thread_client(api_key)

    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=user_content,
                config={"system_instruction": system_content},
            )
            return resp.text.strip() if resp and resp.text else ""
        except exceptions.ResourceExhausted:
            wait_time = (2 ** attempt) * 5
            print(f"\n[Rate Limit] sleep {wait_time}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait_time)
        except exceptions.InvalidArgument as e:
            print(f"\n[Invalid Request] {e}")
            return None
        except Exception as e:
            print(f"\n[Error] {e}")
            time.sleep(5)

    return None


# =============================================================================
# JSON PARSING
# =============================================================================

def parse_llm_json_response(response_text: str) -> Optional[Dict[str, Any]]:
    """
    Parse JSON from LLM response, handling common formatting issues.
    
    LLMs sometimes wrap JSON in markdown code blocks or add extra text.
    This function extracts and parses the JSON robustly.
    """
    if not response_text:
        return None
    
    # Try direct parse first
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    # Try extracting from markdown code block
    # Matches ```json ... ``` or ``` ... ```
    code_block_pattern = r'```(?:json)?\s*\n?(.*?)\n?```'
    matches = re.findall(code_block_pattern, response_text, re.DOTALL)
    
    for match in matches:
        try:
            return json.loads(match.strip())
        except json.JSONDecodeError:
            continue
    
    # Try finding JSON object pattern { ... }
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, response_text, re.DOTALL)
    
    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue
    
    # Failed to parse
    return None


def validate_evaluation_response(parsed: Dict[str, Any], variant_type: str = 'snp') -> Dict[str, Any]:
    """
    Validate and normalize the parsed JSON response.
    Fills in defaults for missing fields.
    """
    # Expected fields
    expected_fields_base = {
        'concordance': 'PARSE_ERROR',
        'explanation': '',
        'signal_category': 'UNKNOWN',
        'MLM_category': 'UNKNOWN',
        'primary_signal_mechanism': '',
        'key_signals': '',
        'rationale_mechanism': '',
        'nt_missed': None,
        'notes': ''
    }
    
    # Add indel-specific field
    if variant_type == 'indel':
        expected_fields_base['embedding_impact'] = 'UNKNOWN'
    
    # Valid concordance values
    valid_concordance = {'CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE'}
    valid_signal_category = {'STRONG', 'MODERATE', 'WEAK', 'ABSENT', 'UNKNOWN'}
    valid_embedding_impact = {'HIGH', 'MODERATE', 'LOW', 'NONE', 'UNKNOWN'}
    
    # Build result with defaults
    result = {}
    for field, default in expected_fields_base.items():
        result[field] = parsed.get(field, default)
    
    # Normalize concordance
    if result['concordance'] not in valid_concordance:
        result['concordance'] = 'PARSE_ERROR'
    
    # Normalize signal_category
    if result['signal_category'] not in valid_signal_category:
        result['signal_category'] = 'UNKNOWN'
  
    
    # Normalize embedding_impact for indels
    if variant_type == 'indel' and result.get('embedding_impact') not in valid_embedding_impact:
        result['embedding_impact'] = 'UNKNOWN'
    
    # Ensure nt_missed is boolean or None
    if result['nt_missed'] is not None:
        result['nt_missed'] = bool(result['nt_missed'])
    
    return result


# =============================================================================
# SINGLE VARIANT EVALUATION
# =============================================================================

def evaluate_single_variant_parallel(row, system_prompt, variant_type, model_name, api_key):
    # Same as your evaluate_single_variant, but swap the call func:
    variation_id = row.get('#VariationID', row.name)
    user_prompt = row.get('llm_prompt', '')
    if not user_prompt:
        return {
            '#VariationID': variation_id,
            'concordance': 'ERROR',
            'explanation': 'No prompt available',
            'signal_category': 'UNKNOWN',
            'MLM_category': 'UNKNOWN',
            'primary_signal_mechanism': '',
            'key_signals': '',
            'rationale_mechanism': '',
            'nt_missed': None,
            'notes': 'Missing llm_prompt column',
            'raw_response': None,
            'parse_success': False
        }

    raw_response = call_llm_api_robust_threaded(
        user_content=user_prompt,
        system_content=system_prompt,
        model_name=model_name,
        api_key=api_key,
        max_retries=10
    )

    # Reuse your parsing/validation functions:
    if raw_response is None:
        return {
            '#VariationID': variation_id,
            'concordance': 'API_ERROR',
            'explanation': 'API call failed',
            'signal_category': 'UNKNOWN',
            'MLM_category': 'UNKNOWN',
            'primary_signal_mechanism': '',
            'key_signals': '',
            'rationale_mechanism': '',
            'nt_missed': None,
            'notes': 'API returned None',
            'raw_response': None,
            'parse_success': False
        }

    parsed = parse_llm_json_response(raw_response)
    if parsed is None:
        return {
            '#VariationID': variation_id,
            'concordance': 'PARSE_ERROR',
            'explanation': 'Failed to parse JSON',
            'signal_category': 'UNKNOWN',
            'MLM_category': 'UNKNOWN',
            'primary_signal_mechanism': '',
            'key_signals': '',
            'rationale_mechanism': '',
            'nt_missed': None,
            'notes': f'Raw response: {raw_response[:500]}',
            'raw_response': raw_response,
            'parse_success': False
        }

    validated = validate_evaluation_response(parsed, variant_type)
    validated['#VariationID'] = variation_id
    validated['raw_response'] = raw_response
    validated['parse_success'] = True
    return validated




# =============================================================================
# BATCH EVALUATION
# =============================================================================


def evaluate_variants_parallel(
    df: pd.DataFrame,
    system_prompt: str,
    variant_type: str,
    model_name: str,
    api_key: str,
    max_workers: int = 10,              # <-- key knob
    save_checkpoint_every: int = 50,
    checkpoint_path: str = None,
    resume_from_checkpoint: bool = True,
):
    results = []
    evaluated_ids = set()

    # Resume
    if checkpoint_path and resume_from_checkpoint:
        try:
            ck = pd.read_parquet(checkpoint_path)
            results = ck.to_dict("records")
            evaluated_ids = set(ck["#VariationID"].astype(str))
            print(f"Resumed {len(evaluated_ids)} already done")
        except FileNotFoundError:
            pass

    # Build jobs for remaining rows
    remaining = []
    for row_idx, row in df.iterrows():
        vid = str(row.get("#VariationID", row_idx))
        if vid not in evaluated_ids:
            remaining.append((row_idx, row))

    total_target = len(evaluated_ids) + len(remaining)
    done_counter = 0
    start = time.time()

    # Submit + collect
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_to_vid = {}
        for row_idx, row in remaining:
            vid = str(row.get("#VariationID", row_idx))
            fut = ex.submit(
                evaluate_single_variant_parallel,
                row, system_prompt, variant_type, model_name, api_key
            )
            future_to_vid[fut] = vid

        for fut in as_completed(future_to_vid):
            vid = future_to_vid[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = {
                    "#VariationID": vid,
                    "concordance": "ERROR",
                    "explanation": f"Worker exception: {e}",
                    'signal_category': 'UNKNOWN',
                    'MLM_category': 'UNKNOWN',
                    'primary_signal_mechanism': '',
                    "key_signals": "",
                    "rationale_mechanism": "",
                    "nt_missed": None,
                    "notes": "Worker crashed",
                    "raw_response": None,
                    "parse_success": False,
                }

            results.append(res)
            done_counter += 1

            elapsed = time.time() - start
            rate_per_min = (done_counter / elapsed) * 60 if elapsed > 0 else 0
            status_char = "✓" if res.get("parse_success") else "✗"
            print(f"\r[{len(results)}/{total_target}] {status_char} {vid} | {rate_per_min:.1f}/min",
                  end="", flush=True)

            # checkpoint safely from the main thread only
            if checkpoint_path and (len(results) % save_checkpoint_every == 0):
                pd.DataFrame(results).to_parquet(checkpoint_path)
                print(f"\n[Checkpoint] saved {len(results)} to {checkpoint_path}")

    print(f"\nDone. Total results: {len(results)}")
    out = pd.DataFrame(results)
    if checkpoint_path:
        out.to_parquet(checkpoint_path)
    return out


# =============================================================================
# RESULTS ANALYSIS
# =============================================================================

def summarize_evaluation_results(results_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate summary statistics from evaluation results.
    """
    total = len(results_df)
    if "concordance" not in results_df.columns:
        raise ValueError("concordance column not found in results_df")
    
    summary = {
        'total_variants': total,
        'parse_success_rate': results_df['parse_success'].mean() if 'parse_success' in results_df.columns else None,
        'signal_category_distribution': results_df['signal_category'].value_counts().to_dict(),
        'MLM_category_distribution': results_df['MLM_category'].value_counts().to_dict(),
        'primary_signal_mechanism_distribution': results_df['primary_signal_mechanism'].value_counts().to_dict(),
        'nt_missed_rate': results_df['nt_missed'].mean() if 'nt_missed' in results_df.columns else None,
    }
    
    # Concordance percentages
    concordance_counts = results_df['concordance'].value_counts()
    for cat in ['CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE']:
        count = concordance_counts.get(cat, 0)
        summary[f'pct_{cat.lower()}'] = count / total * 100 if total > 0 else 0
    
    return summary


def print_evaluation_summary(results_df: pd.DataFrame) -> None:
    """
    Print a formatted summary of evaluation results.
    """
    summary = summarize_evaluation_results(results_df)
    
    print("=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Total variants evaluated: {summary['total_variants']}")
    print(f"Parse success rate: {summary['parse_success_rate']*100:.1f}%")
    print()
    print("Concordance Distribution:")
    print("-" * 30)
    for cat in ['CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE']:
        pct = summary.get(f'pct_{cat.lower()}', 0)
        # count = summary['concordance_distribution'].get(cat, 0)
        bar = '█' * int(pct / 2)
        print(f"  {cat:16} {count:4} ({pct:5.1f}%) {bar}")
    
    # Errors
    error_cats = ['API_ERROR', 'PARSE_ERROR', 'ERROR']
    error_count = sum(summary['concordance_distribution'].get(c, 0) for c in error_cats)
    if error_count > 0:
        print(f"\n  Errors: {error_count}")
    
    print()
    print("Signal Category Distribution:")
    print("-" * 30)
    for cat, count in summary['signal_category_distribution'].items():
        pct = count / summary['total_variants'] * 100
        print(f"  {cat:12} {count:4} ({pct:5.1f}%)")
    
    if summary['nt_missed_rate'] is not None:
        print(f"\nNT Missed Rate: {summary['nt_missed_rate']*100:.1f}%")
    
    print()
    print("MLM Category Distribution:")
    print("-" * 30)
    for cat, count in summary['MLM_category_distribution'].items():
        pct = count / summary['total_variants'] * 100
        print(f"  {cat:12} {count:4} ({pct:5.1f}%)")
    
    print("=" * 60)


# =============================================================================
# MAIN EVALUATION PIPELINE
# =============================================================================

def run_evaluation_pipeline(
    df_with_prompts: pd.DataFrame,
    variant_type: str = 'snp',
    output_path: str = None,
    checkpoint_path: str = None,
    model_name: str = MODEL_NAME,
    api_key: str = API_KEY,
    delay: float = 0.5,
    sample_size: int = None
) -> pd.DataFrame:
    """
    Run the full evaluation pipeline.
    
    Args:
        df_with_prompts: DataFrame with 'llm_prompt' column
        variant_type: 'snp' or 'indel'
        output_path: Path to save final results
        checkpoint_path: Path for checkpointing
        model_name: LLM model
        api_key: API key
        delay: Delay between API calls
        sample_size: If set, evaluate only this many variants
    
    Returns:
        DataFrame with evaluation results
    """
    
    
    # Get system prompt
    system_prompt = get_system_prompt(variant_type)
    
    # Sample if requested
    if sample_size and sample_size < len(df_with_prompts):
        df_eval = df_with_prompts.sample(n=sample_size, random_state=42)
        print(f"Sampled {sample_size} variants for evaluation")
    else:
        df_eval = df_with_prompts
    
    print(f"Starting evaluation of {len(df_eval)} {variant_type} variants")
    print(f"Model: {model_name}")
    print(f"Delay between calls: {delay}s")
    print()
    
    # Run evaluation
    results_df = evaluate_variants_parallel(
        df=df_eval,
        system_prompt=system_prompt,
        variant_type=variant_type,
        model_name=model_name,
        api_key=api_key,
        checkpoint_path=checkpoint_path
    )
    
    # Print summary
    # print_evaluation_summary(results_df)
    
    # Save final results
    if output_path:
        results_df.to_parquet(output_path)
        print(f"\nResults saved to: {output_path}")
        
        # Also save CSV for easy viewing
        csv_path = output_path.replace('.parquet', '.csv')
        results_df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")
    
    return results_df


# =============================================================================
# MERGE RESULTS BACK TO ORIGINAL DATA
# =============================================================================

def merge_evaluation_results(
    original_df: pd.DataFrame,
    results_df: pd.DataFrame,
    on: str = '#VariationID'
) -> pd.DataFrame:
    """
    Merge evaluation results back to the original variant DataFrame.
    
    Args:
        original_df: Original DataFrame with variant data
        results_df: Evaluation results DataFrame
        on: Column to merge on
    
    Returns:
        Merged DataFrame
    """
    # Select columns to merge (exclude raw_response to save space)
    merge_cols = [on, 'concordance', 'explanation', 'signal_category', 
                  'primary_mechanism', 'key_signals', 'rationale_mechanism',
                  'nt_missed', 'parse_success']
    
    # Add embedding_impact if present (indels)
    if 'embedding_impact' in results_df.columns:
        merge_cols.append('embedding_impact')
    
    results_slim = results_df[[c for c in merge_cols if c in results_df.columns]].copy()
    
    # Ensure merge column types match
    original_df[on] = original_df[on].astype(str)
    results_slim[on] = results_slim[on].astype(str)
    
    merged = original_df.merge(results_slim, on=on, how='left', suffixes=('', '_eval'))
    
    return merged


# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    import pyarrow.parquet as pq
    from config import PATHS
    
    # Configuration
    PATHS = {
        "animals_indel_signaled": "parquet/animals_indel_signaled.parquet",
        "animals_snp_signaled": "parquet/animals_snp_signaled.parquet",
        "animals_snp_evaluation_results": "parquet/animals_snp_evaluation_results.parquet",
        "animals_snp_evaluation_checkpoint": "parquet/animals_snp_evaluation_checkpoint.parquet",
        "animals_indel_evaluation_checkpoint": "parquet/animals_indel_evaluation_checkpoint_2.parquet",
        "animals_indel_evaluation_results": "parquet/animals_indel_evaluation_results_2.parquet"
    }
    
    API_KEY = "REDACTED_API_KEY"  # Load from env in production
    
    # Load data
    type = 'snp'
    print(f"Loading {type} data...")
    if type == 'snp':
        df = pq.read_table(PATHS["animals_snp_signaled"]).to_pandas()
        checkpoint_path = PATHS["animals_snp_evaluation_checkpoint"]
        output_path = PATHS["animals_snp_evaluation_results"]
    else:
        df = pq.read_table(PATHS["animals_indel_signaled"]).to_pandas()
        checkpoint_path = PATHS["animals_indel_evaluation_checkpoint"]
        output_path = PATHS["animals_indel_evaluation_results"]
    print(f"Loaded {len(df)} {type}s")
    
    # Build prompts
    print("Building prompts...")
    df_with_prompts = build_prompts_table(df, variant_type=type)



    

    
    # Run evaluation (small sample for testing)
    results = run_evaluation_pipeline(
        df_with_prompts=df_with_prompts,
        variant_type=type,
        output_path=output_path,
        checkpoint_path=checkpoint_path,
        model_name="gemini-3-flash-preview",
        api_key=API_KEY,
        delay=0.5,
        # sample_size=50  # Start with small sample
)

Loading snp data...
Loaded 372 snps
Building prompts...
Starting evaluation of 372 snp variants
Model: gemini-3-flash-preview
Delay between calls: 0.5s

[50/372] ✓ 85 | 72.7/min
[Checkpoint] saved 50 to parquet/animals_snp_evaluation_checkpoint.parquet
[100/372] ✓ 172 | 77.4/min
[Checkpoint] saved 100 to parquet/animals_snp_evaluation_checkpoint.parquet
[150/372] ✓ 255 | 79.7/min
[Checkpoint] saved 150 to parquet/animals_snp_evaluation_checkpoint.parquet
[200/372] ✓ 329 | 81.0/min
[Checkpoint] saved 200 to parquet/animals_snp_evaluation_checkpoint.parquet
[250/372] ✓ 408 | 82.9/min
[Checkpoint] saved 250 to parquet/animals_snp_evaluation_checkpoint.parquet
[300/372] ✓ 483 | 84.5/min
[Checkpoint] saved 300 to parquet/animals_snp_evaluation_checkpoint.parquet
[350/372] ✓ 554 | 85.4/min
[Checkpoint] saved 350 to parquet/animals_snp_evaluation_checkpoint.parquet
[372/372] ✓ 575 | 82.2/min
Done. Total results: 372

Results saved to: parquet/animals_snp_evaluation_results.parquet
CSV saved t

In [25]:

df_with_prompts["llm_prompt"].iloc[0]

'## Variant: 0\n**Variant Type:** indel\n**Variant Size:** 0\n\n### ClinVar Rationale\nMäkeläinen et al. (2019): "ENSCAFT00000005367 exon28 4170dupC* V1390fs" Importantly, the same authors also explain in Table S3 that "*The insertion is found in a cytosine mononucleotide-repeat region [55,146,550–55,146,556] and it is arbitrary at what position the actual insertion occurs. In the text we have denoted the insertion as c.4176insC, p.F1393Lfs*1395)." Variant information changed to reflect HGVS nomenclature\n\n### BED Feature Signals (Genomic Annotations)\n**Top by magnitude:** start_codon=0.028(ref=0.113); splice_donor=0.023(ref=0.201); CTCF-bound=0.017(ref=0.199); always_on_exon=-0.015(ref=0.466); 5UTR+=-0.014(ref=0.331); intron=-0.013(ref=0.411); enhancer_Tissue_invariant=0.012(ref=0.204); 3UTR-=-0.011(ref=0.094); exon=-0.011(ref=0.728); ORF=0.010(ref=0.707)\n**Top gains (positive delta):** start_codon=0.028(ref=0.113); splice_donor=0.023(ref=0.201); CTCF-bound=0.017(ref=0.199); enhanc

# A BATCH API access for LLM evaluation

In [ ]:
import pandas as pd
import json
import time
import re
from typing import Optional, Dict, Any, List
from pathlib import Path
from google.genai import types
from google import genai as genai_client




# =============================================================================
# BATCH FILE PREPARATION
# =============================================================================

def prepare_batch_jsonl(
    df_with_prompts: pd.DataFrame,
    system_prompt: str,
    output_path: str,
    variant_type: str = 'snp',
    model: str = "gemini-3-flash-preview"
) -> str:
    """
    Prepare JSONL file for batch API submission.
    
    Each line follows the format required by Gemini Batch API:
    {"request": {"contents": [...], "system_instruction": {...}}, "custom_id": "..."}
    
    Args:
        df_with_prompts: DataFrame with 'llm_prompt' and '#VariationID' columns
        system_prompt: System instruction text
        output_path: Path for output JSONL file
        variant_type: 'snp' or 'indel'
    
    Returns:
        Path to created JSONL file
    """
    batch_requests = []
    
    for idx, row in df_with_prompts.iterrows():
        variation_id = str(row.get('#VariationID', row.name))
        user_prompt = row.get('llm_prompt', '')
        
        if not user_prompt:
            continue
        
        # Build request in Gemini batch format
        request = {
            "custom_id": variation_id,
            "request": {
                "model": f'models/{model}',
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": user_prompt}]
                    }
                ],
                "system_instruction": {
                    "parts": [{"text": system_prompt}]
                },
                "generation_config": {
                    "temperature": 0.1,  # Low temperature for consistent JSON
                    "max_output_tokens": 4096,
                    "response_mime_type": "application/json"  # Request JSON output
                }
            }
        }
        
        batch_requests.append(request)
    
    # Write JSONL
    with open(output_path, 'w') as f:
        for req in batch_requests:
            f.write(json.dumps(req) + '\n')
    
    print(f"Created batch file: {output_path}")
    print(f"Total requests: {len(batch_requests)}")
    
    return output_path


def prepare_batch_jsonl_simple(
    df_with_prompts: pd.DataFrame,
    system_prompt: str,
    output_path: str
) -> str:
    """
    Simpler JSONL format - just the essentials.
    Adjust based on actual Gemini Batch API requirements.
    """
    with open(output_path, 'w') as f:
        for idx, row in df_with_prompts.iterrows():
            variation_id = str(row.get('#VariationID', row.name))
            user_prompt = row.get('llm_prompt', '')
            
            if not user_prompt:
                continue
            
            entry = {
                "custom_id": variation_id,
                "body": {
                    "contents": [
                        {"role": "user", "parts": [{"text": user_prompt}]}
                    ],
                    "systemInstruction": {
                        "parts": [{"text": system_prompt}]
                    },
                    "generationConfig": {
                        "temperature": 0.1,
                        "maxOutputTokens": 1024
                    }
                }
            }
            
            f.write(json.dumps(entry) + '\n')
    
    print(f"Created batch file: {output_path}")
    return output_path


# =============================================================================
# BATCH JOB MANAGEMENT
# =============================================================================

def upload_and_submit_batch(
    jsonl_path: str,
    display_name: str,
    api_key: str = None,
    model: str = "gemini-3-flash-preview"
) -> Dict[str, Any]:
    """
    Upload JSONL file and submit batch job.
    
    Args:
        jsonl_path: Path to prepared JSONL file
        display_name: Name for the batch job
        api_key: API key
        model: Model to use
    
    Returns:
        Dict with batch job info
    """
    client = genai_client.Client(api_key=api_key)
    
    # Upload the file
    print(f"Uploading {jsonl_path}...")
    uploaded_file = client.files.upload(
    file=jsonl_path,
    config=types.UploadFileConfig(mime_type="application/jsonl")
    )
    print(f"Uploaded: {uploaded_file.name}")
    
    # Create batch job
    print(f"Creating batch job...")
    batch_job = client.batches.create(
        model=model,
        src=uploaded_file.name,
        config={'display_name': display_name}
    )
    
    print(f"Batch job created: {batch_job.name}")
    print(f"Status: {batch_job.state}")
    
    return {
        'job_name': batch_job.name,
        'file_name': uploaded_file.name,
        'display_name': display_name,
        'state': str(batch_job.state),
        'created_time': time.time()
    }


def check_batch_status(
    job_name: str,
    api_key: str = None
) -> Dict[str, Any]:
    """
    Check status of a batch job.
    
    Args:
        job_name: Name of the batch job
        api_key: API key
    
    Returns:
        Dict with status info
    """
    client = genai_client.Client(api_key=api_key)
    
    batch_job = client.batches.get(name=job_name)
    
    status = {
        'name': batch_job.name,
        'state': str(batch_job.state),
        'display_name': getattr(batch_job, 'display_name', ''),
    }
    
    # Add progress info if available
    if hasattr(batch_job, 'request_counts'):
        counts = batch_job.request_counts
        status['total_requests'] = getattr(counts, 'total', 0)
        status['succeeded'] = getattr(counts, 'succeeded', 0)
        status['failed'] = getattr(counts, 'failed', 0)
        status['pending'] = getattr(counts, 'pending', 0)
    
    return status


def wait_for_batch_completion(
    job_name: str,
    api_key: str = None,
    poll_interval: int = 60,
    timeout: int = 7200  # 2 hours
) -> Dict[str, Any]:
    """
    Wait for batch job to complete, polling periodically.
    
    Args:
        job_name: Name of the batch job
        api_key: API key
        poll_interval: Seconds between status checks
        timeout: Maximum seconds to wait
    
    Returns:
        Final status dict
    """
    start_time = time.time()
    
    print(f"Waiting for batch job: {job_name}")
    print(f"Polling every {poll_interval}s, timeout {timeout}s")
    
    while True:
        status = check_batch_status(job_name, api_key)
        # state = status['state']
        state = str(status['state'])
        print(state)
        
        elapsed = time.time() - start_time
        
        # Progress display
        if 'total_requests' in status:
            succeeded = status.get('succeeded', 0)
            total = status.get('total_requests', 0)
            pct = (succeeded / total * 100) if total > 0 else 0
            print(f"\r[{elapsed/60:.1f}min] State: {state} | "
                  f"Progress: {succeeded}/{total} ({pct:.1f}%)", end='', flush=True)
        else:
            print(f"\r[{elapsed/60:.1f}min] State: {state}", end='\n', flush=True)
        
        # Check completion states
        if state in ['JobState.JOB_STATE_SUCCEEDED', 'JOB_STATE_SUCCEEDED', 'SUCCEEDED', 'STATE_SUCCEEDED']:
            print(f"\n✓ Batch job completed successfully!")
            return status
        
        if state in ['JOB_STATE_FAILED', 'FAILED', 'STATE_FAILED']:
            print(f"\n✗ Batch job failed!")
            return status
        
        if state in ['JOB_STATE_CANCELLED', 'CANCELLED', 'STATE_CANCELLED']:
            print(f"\n✗ Batch job was cancelled!")
            return status
        
        # Check timeout
        if elapsed > timeout:
            print(f"\n⚠ Timeout reached ({timeout}s)")
            return status
        
        time.sleep(poll_interval)


def download_batch_results(job_name: str, output_path: str, api_key: str = None) -> str:
    client = genai_client.Client(api_key=api_key)
    
    # 1. Retrieve the job
    batch_job = client.batches.get(name=job_name)
    
    # 2. Extract the filename from the .dest object
    # Based on your diagnostic, the attribute is 'file_name'
    result_file_name = None
    
    if hasattr(batch_job, 'dest') and batch_job.dest:
        result_file_name = batch_job.dest.file_name
        
    # Safety check if it's still None
    if not result_file_name:
        raise ValueError(f"Could not find 'file_name' in batch_job.dest. Object dump: {batch_job}")

    print(f"Downloading results from: {result_file_name}")
    
    # 3. Download using the 'file' keyword argument
    # This returns the raw bytes
    result_bytes = client.files.download(file=result_file_name)
    
    # 4. Save to disk
    with open(output_path, 'wb') as f:
        f.write(result_bytes)
    
    print(f"✓ Results saved to: {output_path}")
    return output_path


# =============================================================================
# RESULTS PARSING
# =============================================================================

def parse_llm_json_response(response_text: str) -> Optional[Dict[str, Any]]:
    """
    Parse JSON from LLM response, handling common formatting issues.
    """
    if not response_text:
        return None
    
    # Try direct parse first
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    # Try extracting from markdown code block
    code_block_pattern = r'```(?:json)?\s*\n?(.*?)\n?```'
    matches = re.findall(code_block_pattern, response_text, re.DOTALL)
    
    for match in matches:
        try:
            return json.loads(match.strip())
        except json.JSONDecodeError:
            continue
    
    # Try finding JSON object pattern { ... }
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, response_text, re.DOTALL)
    
    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue
    
    return None


def parse_batch_results(
    results_path: str,
    variant_type: str = 'snp'
) -> pd.DataFrame:
    """
    Parse batch results JSONL file into DataFrame.
    
    Args:
        results_path: Path to results JSONL file
        variant_type: 'snp' or 'indel' (affects expected fields)
    
    Returns:
        DataFrame with parsed evaluation results
    """
    results = []
    
    with open(results_path, 'r') as f:
        for line_num, line in enumerate(f):
            if not line.strip():
                continue
            
            try:
                entry = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Warning: Failed to parse line {line_num}: {e}")
                continue
            
            # Extract custom_id (VariationID)
            custom_id = entry.get('custom_id', f'unknown_{line_num}')
            
            # Extract response
            response = entry.get('response', {})
            
            # Check for errors
            if 'error' in response:
                results.append({
                    '#VariationID': custom_id,
                    'concordance': 'API_ERROR',
                    'explanation': str(response.get('error', '')),
                    'signal_category': 'UNKNOWN',
                    'MLM_category': 'UNKNOWN',
                    'primary_signal_mechanism': '',
                    'key_signals': '',
                    'rationale_mechanism': '',
                    'nt_missed': None,
                    'notes': 'Missing llm_prompt column',
                    'raw_response': json.dumps(response),
                    'parse_success': False
                })
                continue
            
            # Extract text from response
            # Structure varies - adapt based on actual API response format
            try:
                # Try common response structures
                if 'candidates' in response:
                    text = response['candidates'][0]['content']['parts'][0]['text']
                elif 'content' in response:
                    text = response['content']['parts'][0]['text']
                elif 'text' in response:
                    text = response['text']
                elif 'body' in entry:
                    # Alternative format
                    body = entry['body']
                    if 'candidates' in body:
                        text = body['candidates'][0]['content']['parts'][0]['text']
                    else:
                        text = str(body)
                else:
                    text = json.dumps(response)
            except (KeyError, IndexError, TypeError) as e:
                results.append({
                    '#VariationID': custom_id,
                    'concordance': 'PARSE_ERROR',
                    'explanation': f'Could not extract text: {e}',
                    'signal_category': 'UNKNOWN',
                    'MLM_category': 'UNKNOWN',
                    'primary_signal_mechanism': '',
                    'key_signals': '',
                    'rationale_mechanism': '',
                    'nt_missed': None,
                    'notes': 'Missing llm_prompt column',
                    'raw_response': json.dumps(entry),
                    'parse_success': False
                })
                continue
            
            # Parse the JSON from the text
            parsed = parse_llm_json_response(text)
            
            if parsed is None:
                results.append({
                    '#VariationID': custom_id,
                    'concordance': 'PARSE_ERROR',
                    'explanation': 'Failed to parse JSON from response',
                    'signal_category': 'UNKNOWN',
                    'MLM_category': 'UNKNOWN',
                    'primary_signal_mechanism': '',
                    'key_signals': '',
                    'rationale_mechanism': '',
                    'nt_missed': None,
                    'notes': 'Missing llm_prompt column',
                    'raw_response': text,
                    'parse_success': False
                })
                continue
            
            # Validate and normalize
            result = validate_evaluation_response(parsed, variant_type)
            result['#VariationID'] = custom_id
            result['raw_response'] = text
            result['parse_success'] = True
            
            results.append(result)
    
    df = pd.DataFrame(results)
    print(f"Parsed {len(df)} results from {results_path}")
    
    # Summary
    if 'parse_success' in df.columns:
        success_rate = df['parse_success'].mean() * 100
        print(f"Parse success rate: {success_rate:.1f}%")
    
    if 'concordance' in df.columns:
        print("\nConcordance distribution:")
        print(df['concordance'].value_counts())
    
    return df


def validate_evaluation_response(parsed: Dict[str, Any], variant_type: str = 'snp') -> Dict[str, Any]:
    """
    Validate and normalize the parsed JSON response.
    """
    expected_fields = {
        'concordance': 'PARSE_ERROR',
        'explanation': '',
        'signal_category': 'UNKNOWN',
        'MLM_category': 'UNKNOWN',
        'primary_signal_mechanism': '',
        'key_signals': '',
        'rationale_mechanism': '',
        'nt_missed': None,
        'notes': ''
    }
    
    if variant_type == 'indel':
        expected_fields['embedding_impact'] = 'UNKNOWN'
    
    valid_concordance = {'CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE'}
    valid_signal_category = {'STRONG', 'MODERATE', 'WEAK', 'ABSENT', 'UNKNOWN'}
    
    result = {}
    for field, default in expected_fields.items():
        result[field] = parsed.get(field, default)
    
    if result['concordance'] not in valid_concordance:
        result['concordance'] = 'PARSE_ERROR'
    
    if result['signal_category'] not in valid_signal_category:
        result['signal_category'] = 'UNKNOWN'
    
    if result['nt_missed'] is not None:
        result['nt_missed'] = bool(result['nt_missed'])
    
    return result


# =============================================================================
# COMPLETE BATCH PIPELINE
# =============================================================================

def run_batch_evaluation_pipeline(
    df_with_prompts: pd.DataFrame,
    system_prompt: str,
    variant_type: str = 'snp',
    batch_name: str = 'variant_evaluation',
    work_dir: str = './batch_eval',
    api_key: str = None,
    model: str = "gemini-3-flash-preview",
    wait_for_completion: bool = True,
    poll_interval: int = 60
) -> Optional[pd.DataFrame]:
    """
    Run complete batch evaluation pipeline.
    
    Args:
        df_with_prompts: DataFrame with 'llm_prompt' column
        system_prompt: System instruction
        variant_type: 'snp' or 'indel'
        batch_name: Name for the batch job
        work_dir: Directory for intermediate files
        api_key: API key
        model: Model to use
        wait_for_completion: If True, wait and return results
        poll_interval: Seconds between status checks
    
    Returns:
        DataFrame with results if wait_for_completion, else None
    """
    # Create work directory
    work_path = Path(work_dir)
    work_path.mkdir(parents=True, exist_ok=True)
    
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    
    # Paths
    jsonl_path = work_path / f'{batch_name}_{timestamp}_requests.jsonl'
    results_path = work_path / f'{batch_name}_{timestamp}_results.jsonl'
    job_info_path = work_path / f'{batch_name}_{timestamp}_job_info.json'
    
    # Step 1: Prepare JSONL
    print("=" * 60)
    print("STEP 1: Preparing batch request file")
    print("=" * 60)
    prepare_batch_jsonl(
        df_with_prompts=df_with_prompts,
        system_prompt=system_prompt,
        output_path=str(jsonl_path),
        variant_type=variant_type,
        model=model
    )
    
    # Step 2: Upload and submit
    print("\n" + "=" * 60)
    print("STEP 2: Uploading and submitting batch job")
    print("=" * 60)
    job_info = upload_and_submit_batch(
        jsonl_path=str(jsonl_path),
        display_name=f'{batch_name}_{timestamp}',
        api_key=api_key,
        model=model
    )
    
    # Save job info for later reference
    with open(job_info_path, 'w') as f:
        json.dump(job_info, f, indent=2)
    print(f"Job info saved to: {job_info_path}")
    
    if not wait_for_completion:
        print("\nBatch job submitted. Use check_batch_status() to monitor progress.")
        print(f"Job name: {job_info['job_name']}")
        return None
    
    # Step 3: Wait for completion
    print("\n" + "=" * 60)
    print("STEP 3: Waiting for batch completion")
    print("=" * 60)
    final_status = wait_for_batch_completion(
        job_name=job_info['job_name'],
        api_key=api_key,
        poll_interval=poll_interval
    )
    
    if final_status['state'] not in ['JobState.JOB_STATE_SUCCEEDED','JOB_STATE_SUCCEEDED', 'SUCCEEDED', 'STATE_SUCCEEDED']:
        print(f"\nBatch job did not complete successfully: {final_status['state']}")
        return None
    
    # Step 4: Download results
    print("\n" + "=" * 60)
    print("STEP 4: Downloading results")
    print("=" * 60)
    download_batch_results(
        job_name=job_info['job_name'],
        output_path=str(results_path),
        api_key=api_key
    )
    
    # Step 5: Parse results
    print("\n" + "=" * 60)
    print("STEP 5: Parsing results")
    print("=" * 60)
    results_df = parse_batch_results(
        results_path=str(results_path),
        variant_type=variant_type
    )
    
    # Save parsed results
    parquet_path = work_path / f'{batch_name}_{timestamp}_parsed.parquet'
    results_df.to_parquet(parquet_path)
    print(f"\nParsed results saved to: {parquet_path}")
    
    # Print summary
    print_evaluation_summary(results_df)
    
    return results_df


# =============================================================================
# UTILITY: Resume from job name
# =============================================================================

def resume_batch_evaluation(
    job_name: str,
    output_path: str,
    variant_type: str = 'snp',
    api_key: str = None,
    wait_if_pending: bool = True
) -> pd.DataFrame:
    """
    Resume/retrieve results from an existing batch job.
    
    Useful if you submitted a job and closed your session.
    
    Args:
        job_name: Name of the batch job
        output_path: Path to save results
        variant_type: 'snp' or 'indel'
        api_key: API key
        wait_if_pending: If True, wait for completion if job is still running
    
    Returns:
        DataFrame with results
    """
    # Check status
    status = check_batch_status(job_name, api_key)
    print(f"Job: {job_name}")
    print(f"State: {status['state']}\n")
    
    if status['state'] not in ['JOB_STATE_SUCCEEDED', 'SUCCEEDED', 'STATE_SUCCEEDED']:
        if wait_if_pending:
            print("Job not complete. Waiting...")
            status = wait_for_batch_completion(job_name, api_key)
        else:
            print("Job not complete yet.")
            return None
    
    # Download and parse
    results_jsonl = output_path.replace('.parquet', '_raw.jsonl')
    download_batch_results(job_name, results_jsonl, api_key)
    
    results_df = parse_batch_results(results_jsonl, variant_type)
    results_df.to_parquet(output_path)
    
    return results_df


# =============================================================================
# SUMMARY FUNCTIONS
# =============================================================================

def print_evaluation_summary(results_df: pd.DataFrame) -> None:
    """Print formatted summary of evaluation results."""
    total = len(results_df)
    
    print("\n" + "=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Total variants evaluated: {total}")
    
    if 'parse_success' in results_df.columns:
        success_rate = results_df['parse_success'].mean() * 100
        print(f"Parse success rate: {success_rate:.1f}%")
    
    print("\nConcordance Distribution:")
    print("-" * 40)
    
    concordance_counts = results_df['concordance'].value_counts()
    for cat in ['CONCORDANT', 'PARTIAL', 'DISCORDANT', 'NOT_APPLICABLE', 'PARSE_ERROR', 'API_ERROR']:
        count = concordance_counts.get(cat, 0)
        pct = count / total * 100 if total > 0 else 0
        bar = '█' * int(pct / 2)
        print(f"  {cat:16} {count:5} ({pct:5.1f}%) {bar}")
    
    if 'signal_category' in results_df.columns:
        print("\nSignal Category Distribution:")
        print("-" * 40)
        for cat, count in results_df['signal_category'].value_counts().items():
            pct = count / total * 100
            print(f"  {cat:12} {count:5} ({pct:5.1f}%)")
    
    if 'nt_missed' in results_df.columns:
        missed_rate = results_df['nt_missed'].mean()
        if not pd.isna(missed_rate):
            print(f"\nNT Missed Rate: {missed_rate*100:.1f}%")
    
    print("=" * 60)


# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    import pyarrow.parquet as pq
    from config import PATHS
    
    # Configuration
    PATHS = {
        "animals_indel_signaled": "parquet/animals_indel_signaled.parquet",
        "animals_snp_signaled": "parquet/animals_snp_signaled.parquet",
        "animals_snp_evaluation_results": "parquet/animals_snp_evaluation_results_batch.parquet",
        "animals_snp_evaluation_checkpoint": "parquet/animals_snp_evaluation_checkpoint_batch.parquet",
        "animals_indel_evaluation_checkpoint": "parquet/animals_indel_evaluation_checkpoint_batch.parquet",
        "animals_indel_evaluation_results": "parquet/animals_indel_evaluation_results_batch.parquet"
    }
    
    API_KEY = "REDACTED_API_KEY"  # Load from env in production
    
    # Load data
    type = 'indel'
    print(f"Loading {type} data...")
    if type == 'snp':
        df = pq.read_table(PATHS["animals_snp_signaled"]).to_pandas()
        checkpoint_path = PATHS["animals_snp_evaluation_checkpoint"]
        output_path = PATHS["animals_snp_evaluation_results"]
    else:
        df = pq.read_table(PATHS["animals_indel_signaled"]).to_pandas()
        checkpoint_path = PATHS["animals_indel_evaluation_checkpoint"]
        output_path = PATHS["animals_indel_evaluation_results"]
    print(f"Loaded {len(df)} {type}s")
    
    # Build prompts
    print("Building prompts...")
    df_with_prompts = build_prompts_table(df, variant_type=type)




    system_prompt = get_system_prompt(variant_type='snp')
        
        # Run batch evaluation
    results = run_batch_evaluation_pipeline(
            df_with_prompts=df_with_prompts,
            system_prompt=system_prompt,
            variant_type=type,
            batch_name=f'{type}_concordance_eval_errors_correction',
            work_dir='./batch_eval/snps',
            api_key=API_KEY,
            model="gemini-3-flash-preview",
            wait_for_completion=True,
            poll_interval=60
        )

#     results_df = resume_batch_evaluation(
#     job_name='batches/ew0it9betvhjoeplulrewl1muznu2o9wctpr',
#     output_path='batch_eval/snps/manual_results.parquet',
#     variant_type='snp',
#     api_key=API_KEY
# )
    
    # if results is not None:
    #     # Merge back to original data
    #     snp_with_eval = snp_df.merge(
    #         results[['#VariationID', 'concordance', 'explanation', 'signal_category', 
    #                  'primary_mechanism', 'nt_missed']],
    #         on='#VariationID',
    #         how='left'
    #     )
    #     snp_with_eval.to_parquet('parquet/snps_with_evaluation.parquet')
    #     print("Merged results saved!")
    
    # --- If you need to resume a job later ---
    # results = resume_batch_evaluation(
    #     job_name='batches/your-job-id',
    #     output_path='batch_eval/snps/resumed_results.parquet',
    #     variant_type='snp',
    #     api_key=API_KEY
    # )

Loading indel data...
Loaded 208 indels
Building prompts...
STEP 1: Preparing batch request file
Created batch file: batch_eval\snps\indel_concordance_eval_errors_correction_20260304_153827_requests.jsonl
Total requests: 208

STEP 2: Uploading and submitting batch job
Uploading batch_eval\snps\indel_concordance_eval_errors_correction_20260304_153827_requests.jsonl...
Uploaded: files/ee1s3gymly3d
Creating batch job...
Batch job created: batches/ju2kudqnjnr2k2vf3ycbjnulglbq2jxgq00l
Status: JobState.JOB_STATE_PENDING
Job info saved to: batch_eval\snps\indel_concordance_eval_errors_correction_20260304_153827_job_info.json

STEP 3: Waiting for batch completion
Waiting for batch job: batches/ju2kudqnjnr2k2vf3ycbjnulglbq2jxgq00l
Polling every 60s, timeout 7200s
JobState.JOB_STATE_PENDING
[0.0min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[1.1min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[2.1min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDIN

In [5]:
# 1. Get the raw job object
client = genai_client.Client(api_key=API_KEY)
job_name = 'batches/ew0it9betvhjoeplulrewl1muznu2o9wctpr' # Your specific job ID
batch_job = client.batches.get(name=job_name)

# 2. Print key attributes to find the file name
print("--- Job Inspection ---")
print(f"State: {batch_job.state}")

print("\n[Checking .dest]")
if hasattr(batch_job, 'dest') and batch_job.dest:
    print(f"Type: {type(batch_job.dest)}")
    print(f"Dir: {dir(batch_job.dest)}") # What properties does it have?
else:
    print("batch_job.dest is None")

print("\n[Checking .output_info]")
if hasattr(batch_job, 'output_info') and batch_job.output_info:
    print(f"output_info: {batch_job.output_info}")
else:
    print("batch_job.output_info is None")

# 3. If we see a 'name' in the print output, we can grab it manually:
# manual_file_name = "files/..."

--- Job Inspection ---
State: JobState.JOB_STATE_SUCCEEDED

[Checking .dest]
Type: <class 'google.genai.types.BatchJobDestination'>
Dir: ['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_

In [3]:
client = genai_client.Client(api_key=API_KEY)


In [6]:
client.batches.list()

In [12]:
for job in client.batches.list():
    print(job.name, job.state)

batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk JobState.JOB_STATE_PENDING
batches/l9h7odtcnmcy1uwecgfztsfx2wtt4xmw2q8d JobState.JOB_STATE_SUCCEEDED
batches/bscran8ue1i8mlf70isp23len3pzoxfowz7t JobState.JOB_STATE_SUCCEEDED


In [30]:
job = client.batches.get(name="batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk")


In [31]:
job

BatchJob(
  create_time=datetime.datetime(2026, 2, 27, 6, 55, 42, 192013, tzinfo=TzInfo(0)),
  display_name='snp_concordance_eval_errors_correction_20260227_085533',
  model='models/gemini-3-flash-preview',
  name='batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk',
  state=<JobState.JOB_STATE_PENDING: 'JOB_STATE_PENDING'>,
  update_time=datetime.datetime(2026, 2, 27, 6, 55, 42, 192013, tzinfo=TzInfo(0))
)

In [27]:
results_df = resume_batch_evaluation(
    job_name='batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk',
    output_path='batch_eval/snps/resumed_results.parquet',
    variant_type='snp',
    api_key=API_KEY
)

Job: batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk
State: JobState.JOB_STATE_PENDING

Job not complete. Waiting...
Waiting for batch job: batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk
Polling every 60s, timeout 7200s
JobState.JOB_STATE_PENDING
[0.0min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[1.0min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[2.1min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[3.1min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[4.1min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[5.1min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[6.2min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[7.2min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[8.2min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[9.2min] State: JobState.JOB_STATE_PENDING
JobState.JOB_STATE_PENDING
[10.3min] State: JobState.JOB_STATE_PENDING
JobState.JOB_S

ValueError: Could not find 'file_name' in batch_job.dest. Object dump: name='batches/qkj96s8v03eoaqqehyyepq9o8vea4kgoq1pk' display_name='snp_concordance_eval_errors_correction_20260227_085533' state=<JobState.JOB_STATE_PENDING: 'JOB_STATE_PENDING'> error=None create_time=datetime.datetime(2026, 2, 27, 6, 55, 42, 192013, tzinfo=TzInfo(0)) start_time=None end_time=None update_time=datetime.datetime(2026, 2, 27, 6, 55, 42, 192013, tzinfo=TzInfo(0)) model='models/gemini-3-flash-preview' src=None dest=None completion_stats=None

In [ ]:
submission_df = pd.read_csv("submission_df.csv")

In [39]:
pd.set_option("max_colwidth", 300)
indel_res.columns

Index(['concordance', 'explanation', 'signal_category', 'MLM_category',
       'primary_signal_mechanism', 'key_signals', 'rationale_mechanism',
       'nt_missed', 'notes', 'embedding_impact', '#VariationID',
       'raw_response', 'parse_success'],
      dtype='object')

In [40]:
variants

NameError: name 'variants' is not defined

In [34]:
df["variant_type"].value_counts()

variant_type
snp    372
Name: count, dtype: int64

In [35]:
indels_df = pd.read_parquet("parquet/animals_indel_signaled.parquet")

In [37]:
indels_df["LLR"]

0     NaN
1     NaN
2     NaN
3     NaN
4     NaN
       ..
203   NaN
204   NaN
205   NaN
206   NaN
207   NaN
Name: LLR, Length: 208, dtype: float64